# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
from google.colab import userdata

hf_token = userdata.get("FlyRank")
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [21]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT report_date) AS unique_dates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬──────────────┐
│ total_rows │ unique_content │ unique_dates │
│   int64    │     int64      │    int64     │
├────────────┼────────────────┼──────────────┤
│   78835655 │         427292 │          520 │
└────────────┴────────────────┴──────────────┘

In [22]:
# Verify that every content_hash_id exists across the full reporting period
con.sql(f"""
SELECT
    content_hash_id,
    COUNT(*) AS rows_per_content
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY content_hash_id
ORDER BY rows_per_content DESC
LIMIT 10;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬──────────────────┐
│     content_hash_id      │ rows_per_content │
│         varchar          │      int64       │
├──────────────────────────┼──────────────────┤
│ content_e79531cf26443659 │              520 │
│ content_a8b8771f27de20f6 │              520 │
│ content_0672d8db776419c0 │              520 │
│ content_17c76438024dece9 │              520 │
│ content_f3a75d8cf58dd50b │              520 │
│ content_0642dc7f62d4f780 │              520 │
│ content_5e770041ee8f2231 │              520 │
│ content_5175438fecb054a4 │              520 │
│ content_de7b08874af74c00 │              520 │
│ content_b7182f464dcd1e73 │              520 │
├──────────────────────────┴──────────────────┤
│ 10 rows                           2 columns │
└─────────────────────────────────────────────┘

In [23]:
con.sql(f"""
SELECT
    month,
    COUNT(DISTINCT report_date) AS days
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY month
ORDER BY month;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────┐
│  month  │ days  │
│ varchar │ int64 │
├─────────┼───────┤
│ 2025-01 │     5 │
│ 2025-02 │    28 │
│ 2025-03 │    31 │
│ 2025-04 │    30 │
│ 2025-05 │    31 │
│ 2025-06 │    30 │
│ 2025-07 │    31 │
│ 2025-08 │    31 │
│ 2025-09 │    30 │
│ 2025-10 │    31 │
│ 2025-11 │    30 │
│ 2025-12 │    31 │
│ 2026-01 │    31 │
│ 2026-02 │    28 │
│ 2026-03 │    31 │
│ 2026-04 │    30 │
│ 2026-05 │    31 │
│ 2026-06 │    30 │
├─────────┴───────┤
│     18 rows     │
└─────────────────┘

**Unit of analysis:**

One row represents one content item (content_hash_id) on one report_date.

**Time window:**

The dataset covers January 2025 through June 2026.
The first month is partial (5 days), while the following months contain their expected number of reporting days.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [25]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.